# Find the bugs: testing and debugging a heat-conduction script

*Python for Engineers: Case Studies from Calculator to Simulation* · Notebook 08 · Yanwei Wang

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktwyw/python-for-engineers-case-studies/blob/main/notebooks/08_Find_the_Bugs.ipynb)

A colleague wrote a short program for the temperature in an aluminium rod. The left end is held at 100 °C by a heater, the right end at 20 °C by a cooled clamp, and the rod starts at room temperature. The program uses the same explicit update you met in notebook 05. Your colleague says: *"It mostly works, but the numbers look a bit odd."*

The script contains **five bugs**. Some crash the program, but most do not: they give wrong numbers that look believable. This notebook is about finding them **systematically** rather than by staring at the code. Each bug is found with a different tool:

| Bug | Tool that finds it |
|---|---|
| 1 | reading Python's error message |
| 2 | checking a physical rule with `assert` |
| 3 | testing a small function against a hand calculation |
| 4 | testing the simplest possible case |
| 5 | validating the inputs and failing loudly |

**How to study:** before reading each explanation, look at the evidence and guess where the bug is. Debugging is a skill you learn by doing it.

<!-- pfe-study-guide -->
## Study guide

| | |
|---|---|
| **Status** | Core · Part 3, Getting it right |
| **Time** | about 45 min (about 75 min with the practice) |
| **Before this** | Notebooks 01–03 (lists, functions) and notebook 05 (the heat-conduction update). |
| **Essential** | All sections. |
| **Optional on a first reading** | — |
| **Next** | 09, Reaction kinetics and parameter fitting. |

**You have understood this notebook if you can, without looking back:**

1. Read a Python error report from the bottom up and find the failing line.
2. Turn a physical rule into an `assert` check, and explain why a passing check does not prove a program correct.
3. Shrink a problem and trace it with `print` to see where it goes wrong.
4. Write a unit test against a hand calculation, including the simplest possible case.
5. Explain why `T_new = T` does not copy a list and why `T[-1]` fails silently.
6. Validate inputs with `raise ValueError` and a helpful message, and keep a suite of tests to re-run after every change.

**Not expected of you:** Interactive debuggers such as `pdb`; `print`, `assert` and tests are enough here.

*The comment on every code line is a reading aid. In your own programs, comment the reasons and the non-obvious steps.*

## 1. The model in brief

The rod is divided into $N$ equally spaced **nodes**, numbered $0$ to $N-1$. Node $0$ is the left end and node $N-1$ the right end. In each time step $\Delta t$, every **interior** node moves towards the average of its neighbours:

$$T_i^{\text{new}} = T_i + r\,\left(T_{i-1} - 2T_i + T_{i+1}\right), \qquad r = \frac{\alpha\,\Delta t}{\Delta x^2}, \qquad \alpha = \frac{k}{\rho\,c_p}.$$

The two end nodes are **not** updated: their temperatures are fixed. The update is only trustworthy if $r \le 1/2$.

We know three facts the correct program must respect. They will become our tests:

1. The end temperatures never change.
2. No temperature can go above the hottest or below the coldest value present at the start. Heat conduction smooths out differences; it never creates new extremes.
3. After a long time the profile becomes a **straight line** from 100 °C to 20 °C.

The colleague's program uses plain Python lists, so every step is visible.

In [1]:
# Rod length in metres.
LENGTH = 0.20
# Number of nodes, including both ends.
N_NODES = 21
# Aluminium: thermal conductivity in W/(m·K).
K_AL = 237.0
# Aluminium: density in kg/m³.
RHO_AL = 2700.0
# Aluminium: specific heat capacity in J/(kg·K).
CP_AL = 900.0
# Fixed end temperatures in °C: heater on the left, cooled clamp on the right.
T_LEFT, T_RIGHT = 100.0, 20.0
# The rod starts at room temperature.
T_START = 20.0
# Time step in seconds.
DT = 0.2
# 3000 steps of 0.2 s simulate 600 s, long enough to reach the steady state.
N_STEPS = 3000

## 2. The script as you received it

Read the code once from top to bottom. Do you spot anything already? Do not fix it yet: in real work you rarely see bugs by reading alone, so we will let the program and the tests show them.

In [2]:
# Distance between neighbouring nodes.
def grid_spacing(length, n_nodes):
    # The rod length shared out over the nodes.
    return length / n_nodes


# The dimensionless number r that controls the update.
def diffusion_number(alpha, dt, dx):
    # r = α Δt / Δx².
    return alpha * dt / dx**2


# Advance every node by one time step.
def step(T, r):
    # The list that will hold the new temperatures.
    T_new = T
    # Visit the nodes.
    for i in range(len(T) - 1):
        # Move node i towards the average of its neighbours.
        T_new[i] = T[i] + r * (T[i - 1] - 2 * T[i] + T[i + 1])
    # Hand back the new temperatures.
    return T_new


# Run the whole simulation and return the final temperatures.
def simulate_rod(length, n_nodes, k, rho, cp, t_left, t_right, t_start, dt, n_steps):
    # Thermal diffusivity in m²/s.
    alpha = k / (rho * Cp)
    # Node spacing in metres.
    dx = grid_spacing(length, n_nodes)
    # The update's controlling number.
    r = diffusion_number(alpha, dt, dx)
    # Every node starts at the initial temperature ...
    T = [t_start] * n_nodes
    # ... except the heated left end ...
    T[0] = t_left
    # ... and the cooled right end.
    T[-1] = t_right
    # Repeat the update n_steps times.
    for _ in range(n_steps):
        # Replace the old temperatures with the new ones.
        T = step(T, r)
    # The temperatures at the final time.
    return T

## 3. Bug 1: read the error message

Run the simulation. The cell below catches the error and prints only its last line, so that this notebook keeps running. Delete the `#` at the start of the last line to see Python's full error report, then put it back.

**Read the report from the bottom up.** The last line names the **type** of error and gives a short explanation. The lines above it show *where* it happened: the cell, the line number and the function, with an arrow at the failing line.

In [3]:
# Try to run the colleague's program.
try:
    # The full 600 s simulation.
    final = simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, CP_AL, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# A NameError means Python met a name it does not know.
except NameError as err:
    # Show the error's type and message.
    print("NameError:", err)
# Delete the # on the next line to see the full report (then restore it).
# simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, CP_AL, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)

NameError: name 'Cp' is not defined


`name 'Cp' is not defined`: the function's parameter is called `cp`, but the formula uses `Cp`. Python treats upper and lower case as different letters. The full report points to the line `alpha = k / (rho * Cp)` inside `simulate_rod`.

This is the **easy** kind of bug: the program stops and tells you where. We fix it by redefining the function with one change, marked `FIXED`.

In [4]:
# Run the whole simulation and return the final temperatures.
def simulate_rod(length, n_nodes, k, rho, cp, t_left, t_right, t_start, dt, n_steps):
    # FIXED (bug 1): the parameter is cp, not Cp.
    alpha = k / (rho * cp)
    # Node spacing in metres.
    dx = grid_spacing(length, n_nodes)
    # The update's controlling number.
    r = diffusion_number(alpha, dt, dx)
    # Every node starts at the initial temperature ...
    T = [t_start] * n_nodes
    # ... except the heated left end ...
    T[0] = t_left
    # ... and the cooled right end.
    T[-1] = t_right
    # Repeat the update n_steps times.
    for _ in range(n_steps):
        # Replace the old temperatures with the new ones.
        T = step(T, r)
    # The temperatures at the final time.
    return T


# Run again.
final = simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, CP_AL, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# Show every node with one decimal.
print([round(t, 1) for t in final])

[20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0, 20.0]


## 4. Bug 2: check a physical rule with `assert`

The program runs without an error, but every node ends at 20 °C, as if the heater had no effect. **Predict:** what should the first number be? Where in the code could the heat be lost?

An `assert` statement turns an expectation into a check: `assert condition, message` does nothing if the condition is true and stops with an `AssertionError` if it is false. We write a function that checks the first two facts from Section 1.

In [5]:
# Check two physical rules on a list of temperatures.
def check_physics(T, t_left, t_right, t_low, t_high):
    # Rule 1: the left end must still be at its fixed value.
    assert T[0] == t_left, f"left end changed: {T[0]:.2f} °C instead of {t_left} °C"
    # Rule 1: the right end must still be at its fixed value.
    assert T[-1] == t_right, f"right end changed: {T[-1]:.2f} °C instead of {t_right} °C"
    # Rule 2: no temperature outside the starting range (with a tiny allowance for round-off).
    assert all(t_low - 1e-9 <= t <= t_high + 1e-9 for t in T), "a temperature left the allowed range"


# Apply the checks to the result above.
try:
    # The starting range runs from 20 °C to 100 °C.
    check_physics(final, T_LEFT, T_RIGHT, 20.0, 100.0)
    # Only reached if every check passed.
    print("all physics checks passed")
# A failed assert raises AssertionError, carrying our message.
except AssertionError as err:
    # Show which rule failed.
    print("AssertionError:", err)

AssertionError: left end changed: 20.00 °C instead of 100.0 °C


The heated end has cooled down, which is impossible: the heater holds it at 100 °C. Something in `step` changes node 0. **Tracing** means printing what the program does, one operation at a time, on a case small enough to follow by hand. We make a copy of `step` with a `print` inside the loop and run it once on five nodes.

In [6]:
# A copy of step that reports every update it makes.
def step_traced(T, r):
    # Same as step: the list that will hold the new temperatures.
    T_new = T
    # Same loop as step.
    for i in range(len(T) - 1):
        # Report which node is updated and which neighbours it uses.
        print(f"i = {i}: left neighbour T[{i - 1}] = {T[i - 1]}, right neighbour T[{i + 1}] = {T[i + 1]}")
        # Same update as step.
        T_new[i] = T[i] + r * (T[i - 1] - 2 * T[i] + T[i + 1])
    # Hand back the result.
    return T_new


# Five nodes: hot left end, everything else at 20 °C.
tiny = [100.0, 20.0, 20.0, 20.0, 20.0]
# One traced step with r = 0.25.
step_traced(tiny, 0.25)

i = 0: left neighbour T[-1] = 20.0, right neighbour T[1] = 20.0
i = 1: left neighbour T[0] = 60.0, right neighbour T[2] = 20.0
i = 2: left neighbour T[1] = 30.0, right neighbour T[3] = 20.0
i = 3: left neighbour T[2] = 22.5, right neighbour T[4] = 20.0


[60.0, 30.0, 22.5, 20.625, 20.0]

The first line of the trace shows the bug: for `i = 0` the "left neighbour" is `T[-1]`, the **last** element of the list. In Python a negative index counts from the end, so `T[-1]` is valid code and raises no error. The left end of the rod is treated as if it touched the right end.

The loop should visit only the **interior** nodes, 1 to $N-2$. `range(1, len(T) - 1)` produces exactly those numbers.

In [7]:
# Advance every interior node by one time step.
def step(T, r):
    # The list that will hold the new temperatures.
    T_new = T
    # FIXED (bug 2): visit only the interior nodes 1 to N-2; the ends stay fixed.
    for i in range(1, len(T) - 1):
        # Move node i towards the average of its neighbours.
        T_new[i] = T[i] + r * (T[i - 1] - 2 * T[i] + T[i + 1])
    # Hand back the new temperatures.
    return T_new


# Run the whole simulation again.
final = simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, CP_AL, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# Apply the same physics checks; they should pass now.
check_physics(final, T_LEFT, T_RIGHT, 20.0, 100.0)
# Report.
print("physics checks passed; midpoint =", round(final[N_NODES // 2], 2), "°C")

physics checks passed; midpoint = 60.0 °C


## 5. Bug 3: test a small function against a hand calculation

The physics checks pass. That does **not** prove the program is correct. A check only finds the kind of error it looks for.

A **unit test** runs one small function on an input small enough to work out by hand, and compares the result with your answer. Take five nodes `[100, 20, 20, 20, 20]` and $r = 0.25$. Using only the **old** values:

- node 1: $20 + 0.25\,(100 - 2\cdot 20 + 20) = 40$;
- node 2: $20 + 0.25\,(20 - 40 + 20) = 20$;
- node 3: $20 + 0.25\,(20 - 40 + 20) = 20$.

So one step must give `[100, 40, 20, 20, 20]`.

In [8]:
# A unit test of step, against the hand calculation above.
def test_step_by_hand():
    # The starting temperatures.
    before = [100.0, 20.0, 20.0, 20.0, 20.0]
    # One step with r = 0.25.
    after = step(before, 0.25)
    # The answer worked out by hand.
    expected = [100.0, 40.0, 20.0, 20.0, 20.0]
    # Compare, showing both lists if they differ.
    assert after == expected, f"got {after}, expected {expected}"


# Run the test.
try:
    # Call the test function.
    test_step_by_hand()
    # Only reached if the test passed.
    print("test_step_by_hand passed")
# Show the failure message.
except AssertionError as err:
    # The message contains both lists.
    print("AssertionError:", err)

AssertionError: got [100.0, 40.0, 25.0, 21.25, 20.0], expected [100.0, 40.0, 20.0, 20.0, 20.0]


Node 1 is right, but nodes 2 and 3 are too warm, as if heat had travelled two nodes in one step. Node 2 behaves as if its left neighbour were already 40 °C, the **new** value of node 1. So `step` is reading values it has just written. The trace in Section 4 already hinted at this: for `i = 1` it reported the left neighbour as 60.0, the value just written to node 0, not the old 100.0.

The cause is the first line of `step`. `T_new = T` does **not** copy the list; it gives the same list a second name. Python names are labels attached to objects, and two labels can be attached to one object. The `is` operator asks whether two names refer to the same object:

In [9]:
# A list with one name.
a = [1.0, 2.0, 3.0]
# A second name for the same list (no copy is made).
b = a
# A real copy: a new list with the same contents.
c = a.copy()
# Change the list through the name b.
b[0] = 99.0
# a shows the change too, because a and b are the same list; c does not.
print("a =", a, " b =", b, " c =", c)
# 'is' confirms it: True for a and b, False for a and c.
print("b is a:", b is a, "  c is a:", c is a)

a = [99.0, 2.0, 3.0]  b = [99.0, 2.0, 3.0]  c = [1.0, 2.0, 3.0]
b is a: True   c is a: False


In [10]:
# Advance every interior node by one time step.
def step(T, r):
    # FIXED (bug 3): copy the list, so new values never overwrite the old ones still needed.
    T_new = T.copy()
    # Visit only the interior nodes 1 to N-2; the ends stay fixed.
    for i in range(1, len(T) - 1):
        # Move node i towards the average of its neighbours, using old values only.
        T_new[i] = T[i] + r * (T[i - 1] - 2 * T[i] + T[i + 1])
    # Hand back the new temperatures.
    return T_new


# The unit test must pass now.
test_step_by_hand()
# Report.
print("test_step_by_hand passed")

test_step_by_hand passed


Look back at the physics check in Section 4: it passed **with this bug still present**. The wrong version still kept the ends fixed and the temperatures in range, and it still reached the right straight line in the end, just by the wrong route. Only a test that knew the exact answer could see the difference.

## 6. Bug 4: test the simplest possible case

Every function deserves a test. For `grid_spacing`, choose a case where the answer is obvious: **three nodes on a 1 m rod** sit at 0 m, 0.5 m and 1 m, so the spacing is 0.5 m.

In [11]:
# A unit test of grid_spacing on a case with an obvious answer.
def test_grid_spacing():
    # Three nodes on a 1 m rod are 0.5 m apart.
    assert grid_spacing(1.0, 3) == 0.5, f"got {grid_spacing(1.0, 3)}, expected 0.5"


# Run the test.
try:
    # Call the test function.
    test_grid_spacing()
    # Only reached if the test passed.
    print("test_grid_spacing passed")
# Show the failure message.
except AssertionError as err:
    # The message contains the wrong value.
    print("AssertionError:", err)

AssertionError: got 0.3333333333333333, expected 0.5


Three nodes have only **two gaps** between them. This is the **fencepost error**: a fence with $N$ posts has $N-1$ gaps. The wrong spacing was 5 % too small for 21 nodes, so $r$ was about 10 % too large and the rod heated about 10 % too fast. No plot would have shown that, and no crash would have warned you.

In [12]:
# Distance between neighbouring nodes.
def grid_spacing(length, n_nodes):
    # FIXED (bug 4): n nodes have n - 1 gaps between them.
    return length / (n_nodes - 1)


# The test must pass now.
test_grid_spacing()
# Report the spacing for the real rod: 0.20 m over 20 gaps.
print("test_grid_spacing passed; dx =", grid_spacing(LENGTH, N_NODES), "m")

test_grid_spacing passed; dx = 0.01 m


## 7. Bug 5: validate the inputs and fail loudly

The code is now correct, but a program can still be **used** wrongly. Your colleague's original run took $c_p$ from a datasheet that gives it in **kJ**/(kg·K), and typed `0.9` instead of `900`. **Predict:** what happens to $\alpha$, and then to $r$?

In [13]:
# The datasheet value, in kJ/(kg·K): the wrong unit for our formula.
cp_from_datasheet = 0.9
# Run the corrected program with the wrong-unit input.
bad = simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, cp_from_datasheet, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# Look at the midpoint.
print("midpoint temperature:", bad[N_NODES // 2])
# Compute r for this input, to see why.
r_bad = diffusion_number(K_AL / (RHO_AL * cp_from_datasheet), DT, grid_spacing(LENGTH, N_NODES))
# A thousand times too large an α gives a thousand times too large an r.
print(f"r = {r_bad:.0f}, far above the limit 0.5")

midpoint temperature: nan
r = 195, far above the limit 0.5


`nan` means *not a number*. The temperatures grew without limit until they overflowed to infinity, and then $\infty - \infty$ gave `nan`. **Python gave no warning**: floating-point arithmetic quietly carries `inf` and `nan` along. A program that can produce garbage without complaint should check its inputs and **refuse to run**.

`raise ValueError(message)` stops the function with an error and a message you write. A good message says what is wrong, gives the value and suggests the likely cause.

In [14]:
# Run the whole simulation and return the final temperatures.
def simulate_rod(length, n_nodes, k, rho, cp, t_left, t_right, t_start, dt, n_steps):
    # Thermal diffusivity in m²/s.
    alpha = k / (rho * cp)
    # Node spacing in metres.
    dx = grid_spacing(length, n_nodes)
    # The update's controlling number.
    r = diffusion_number(alpha, dt, dx)
    # FIXED (bug 5): refuse inputs for which the explicit update is unstable.
    if r > 0.5:
        # Stop with a message that helps the user find the cause.
        raise ValueError(f"unstable time step: r = {r:.3g} > 0.5 (alpha = {alpha:.3g} m²/s). "
                         f"Reduce dt, or check the units of k, rho and cp (cp in J/(kg·K)).")
    # Every node starts at the initial temperature ...
    T = [t_start] * n_nodes
    # ... except the heated left end ...
    T[0] = t_left
    # ... and the cooled right end.
    T[-1] = t_right
    # Repeat the update n_steps times.
    for _ in range(n_steps):
        # Replace the old temperatures with the new ones.
        T = step(T, r)
    # The temperatures at the final time.
    return T


# The wrong-unit input must now be refused.
try:
    # Same call as before.
    simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, cp_from_datasheet, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# The error we raised ourselves.
except ValueError as err:
    # Show the message.
    print("ValueError:", err)

ValueError: unstable time step: r = 195 > 0.5 (alpha = 0.0975 m²/s). Reduce dt, or check the units of k, rho and cp (cp in J/(kg·K)).


## 8. Final checks against answers we know

All five bugs are fixed. Before we trust the program, we compare it with two results that come from **outside** the program:

1. **Steady state:** after 600 s the profile must be the straight line from 100 °C to 20 °C.
2. **Decay of a sine bump:** if the rod starts at $20 + 10\sin(\pi x/L)$ °C with both ends at 20 °C, the bump keeps its shape and its height decays as $10\,e^{-\alpha(\pi/L)^2 t}$. That formula comes from the heat equation (notebook 05, optional section), not from our code.

In [15]:
# The math module provides sin, exp and pi.
import math
# Diffusivity of aluminium in m²/s.
alpha_al = K_AL / (RHO_AL * CP_AL)
# Node spacing of the real rod.
dx = grid_spacing(LENGTH, N_NODES)
# Positions of the nodes, in metres.
x = [i * dx for i in range(N_NODES)]

# Check 1: run to 600 s from the original starting state.
final = simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, CP_AL, T_LEFT, T_RIGHT, T_START, DT, N_STEPS)
# The straight line from 100 °C at x = 0 to 20 °C at x = L.
line = [T_LEFT + (T_RIGHT - T_LEFT) * xi / LENGTH for xi in x]
# The largest difference from the straight line, over all nodes.
worst = max(abs(t - s) for t, s in zip(final, line))
# Report and check: within 0.01 °C.
print(f"check 1: largest deviation from the straight line = {worst:.2e} °C")
# Fail loudly if the steady state is wrong.
assert worst < 0.01

# Check 2: start from the sine bump, both ends at 20 °C.
T = [20.0 + 10.0 * math.sin(math.pi * xi / LENGTH) for xi in x]
# The update number for the real rod.
r = diffusion_number(alpha_al, DT, dx)
# 200 steps of 0.2 s give 40 s.
for _ in range(200):
    # One correct step.
    T = step(T, r)
# The height of the bump at the midpoint, from the simulation.
bump_simulated = T[N_NODES // 2] - 20.0
# The same height from the exact formula.
bump_exact = 10.0 * math.exp(-alpha_al * (math.pi / LENGTH) ** 2 * 40.0)
# Report both.
print(f"check 2: bump at 40 s, simulated {bump_simulated:.3f} °C, exact {bump_exact:.3f} °C")
# Agreement within 0.02 °C.
assert abs(bump_simulated - bump_exact) < 0.02

check 1: largest deviation from the straight line = 2.71e-05 °C
check 2: bump at 40 s, simulated 3.818 °C, exact 3.819 °C


**Keep the tests.** The tests we wrote are now part of the program. When anyone changes the code later, they run all the tests again; a test that used to pass and now fails shows at once that the change broke something (a **regression**). Collecting the tests in one function makes that a single call.

In [16]:
# A unit test: the stability check must reject r > 0.5.
def test_rejects_unstable_input():
    # Expect a ValueError ...
    try:
        # ... from the wrong-unit specific heat.
        simulate_rod(LENGTH, N_NODES, K_AL, RHO_AL, 0.9, T_LEFT, T_RIGHT, T_START, DT, 10)
    # The expected outcome: the test passes.
    except ValueError:
        # Leave the test successfully.
        return
    # Reaching this line means no error was raised: the test fails.
    raise AssertionError("an unstable input was accepted")


# A unit test: the ends stay fixed, even after many steps on a small rod.
def test_ends_fixed():
    # 200 steps on a 5-node rod.
    T = simulate_rod(1.0, 5, K_AL, RHO_AL, CP_AL, 50.0, 10.0, 30.0, 100.0, 200)
    # Both ends exactly unchanged.
    assert T[0] == 50.0 and T[-1] == 10.0


# Run every test and report.
def run_all_tests():
    # The tests, in the order they were written.
    tests = [test_step_by_hand, test_grid_spacing, test_rejects_unstable_input, test_ends_fixed]
    # Call each one; any failure stops here with its message.
    for test in tests:
        # Run one test.
        test()
        # Report it by name.
        print("passed:", test.__name__)


# Run the whole suite.
run_all_tests()

passed: test_step_by_hand
passed: test_grid_spacing
passed: test_rejects_unstable_input
passed: test_ends_fixed


## 9. A strategy you can reuse

| Symptom | First thing to try |
|---|---|
| The program stops with an error | Read the last line of the report, then the arrow showing the line. |
| The numbers break a rule you know | Turn the rule into an `assert`. |
| You cannot see *where* it goes wrong | Shrink the problem (5 nodes, 1 step) and trace it with `print`. |
| A function might be wrong | Test it on an input you can work out by hand. |
| Garbage (`nan`, `inf`, absurd values) appears without an error | Check the inputs and units; add a check that raises an error. |
| You think you have fixed it | Run *all* the tests again, not just the new one. |

Change **one thing at a time** and re-run after each change. If you change three things at once and the result improves, you do not know which change helped, or whether one of the others did harm.

## 10. Practice

The two functions below each contain one bug. For each: write a test that **fails** on the buggy version, explain the bug in one sentence, then fix it and show that the test passes.

1. `mean_temperature(T)` should return the average of all values in `T`.
2. `heat_flux(T, dx, k)` should return Fourier's law, $q = -k\,(T_1 - T_0)/\Delta x$, between the first two nodes. Heat flows from hot to cold, so for a rod that is hotter on the left, $q$ must be **positive** (towards increasing $x$).
3. Why does `b = a` followed by `b.append(5)` change `a`? How do you avoid it?

In [17]:
# The average of all temperatures (contains one bug).
def mean_temperature(T):
    # Sum of the values divided by their number.
    return sum(T[1:]) / len(T)


# Heat flux between the first two nodes, in W/m² (contains one bug).
def heat_flux(T, dx, k):
    # Fourier's law with a forward difference.
    return k * (T[1] - T[0]) / dx

In [18]:
# Your tests and fixes for the practice questions go here.

**Worked answers.** Run the cell after you have tried the questions.

In [19]:
# Q1: a list whose mean is obvious: [10, 20, 30] has mean 20.
print("Q1 buggy:", mean_temperature([10.0, 20.0, 30.0]), "(expected 20.0)")


# Q1 fixed: T[1:] skipped the first value; sum all of them.
def mean_temperature(T):
    # Sum of all values divided by their number.
    return sum(T) / len(T)


# The test now passes.
assert mean_temperature([10.0, 20.0, 30.0]) == 20.0
# Report.
print("Q1 fixed: test passed")

# Q2: hotter on the left (100 °C, then 90 °C, 0.01 m apart) must give a positive flux.
print("Q2 buggy:", heat_flux([100.0, 90.0], 0.01, 237.0), "W/m² (expected positive)")


# Q2 fixed: the minus sign of Fourier's law was missing.
def heat_flux(T, dx, k):
    # q = -k dT/dx, with dT/dx estimated by a forward difference.
    return -k * (T[1] - T[0]) / dx


# The test now passes: 237 × 10 / 0.01 = 237,000 W/m².
assert heat_flux([100.0, 90.0], 0.01, 237.0) > 0
# Report the value.
print("Q2 fixed:", heat_flux([100.0, 90.0], 0.01, 237.0), "W/m²")

# Q3: b = a makes a second name for the same list, so a change through b shows in a.
a = [1, 2, 3]
# b.copy() would make an independent list; here we use it.
b = a.copy()
# Change b only.
b.append(5)
# a is unchanged.
print("Q3: a =", a, " b =", b)

Q1 buggy: 16.666666666666668 (expected 20.0)
Q1 fixed: test passed
Q2 buggy: -237000.0 W/m² (expected positive)
Q2 fixed: 237000.0 W/m²
Q3: a = [1, 2, 3]  b = [1, 2, 3, 5]


## What to take away

- **A program that runs is not a program that is right.** Four of the five bugs gave believable numbers or no error at all.
- Read error reports from the bottom up: error type and message first, then the line.
- Turn what you know about the physics into `assert` checks; a check only finds the errors it looks for.
- Unit-test small functions on inputs you can work out by hand, including the simplest possible case.
- `T_new = T` does not copy a list; use `T.copy()`. A negative index such as `T[-1]` is valid Python and fails silently.
- Validate inputs and `raise` a clear error rather than return garbage. Keep your tests and run them all after every change.

Notebook 09 uses these habits on a harder problem: fitting a reaction-rate model to measured data, and judging whether the fit can be trusted.